# Predicting Student Health Risk - Baseline Modeling

This public notebook builds a leaderboard-ready baseline for **Playground Series S6E7** and writes a notebook-generated `submission.csv`.

This version keeps the v8 **balanced LGBM/XGB domain ensemble** champion, then tests three conservative extensions:

1. **HGB + LGBM/XGB probability blending** for complementary preprocessing paths;
2. **class-prior calibration** with small multipliers for `fit` and `unhealthy`;
3. an ethical **CatBoost signal engine** inspired by public arbitration notebooks.

All experiments use only our own models and OOF probabilities: no external submission pools, no public-score weighting, and no hidden leaderboard feedback.

## 1. Setup

We optimize for reliable public-notebook workflow:

- **row-safe features only**: every feature is computed independently for train and test rows;
- **3-fold stratified OOF validation**: every candidate receives out-of-fold diagnostics without making the public notebook too slow;
- **minority-class guardrails**: local accuracy alone is not trusted after the failed unweighted HGB submission;
- **probability-level experiments**: class multipliers and micro-flips are selected from OOF probabilities, not hard-coded from leaderboard feedback;
- **one final artifact**: the selected model writes `/kaggle/working/submission.csv`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import warnings
import zipfile

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

SEED = 42
N_SPLITS = 3
EPS = 1e-6

REQUIRED_FILES = ['train.csv', 'test.csv', 'sample_submission.csv']


def has_required_files(candidate: Path) -> bool:
    return all((candidate / name).exists() for name in REQUIRED_FILES)


def download_competition_data(target: Path):
    target.mkdir(parents=True, exist_ok=True)
    if has_required_files(target):
        return target

    try:
        subprocess.run(
            ['kaggle', 'competitions', 'download', '-c', 'playground-series-s6e7', '-p', str(target)],
            check=True,
        )
        for zip_path in target.glob('*.zip'):
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(target)
    except Exception as exc:
        print('Competition-data download fallback failed:', type(exc).__name__, exc)

    return target if has_required_files(target) else None


def find_data_dir() -> Path:
    candidates = [
        Path('/kaggle/input/playground-series-s6e7'),
        Path('../input/playground-series-s6e7'),
        Path('data'),
    ]
    for candidate in candidates:
        if has_required_files(candidate):
            return candidate

    input_root = Path('/kaggle/input')
    if input_root.exists():
        for train_path in input_root.rglob('train.csv'):
            candidate = train_path.parent
            if has_required_files(candidate):
                return candidate

    if Path('/kaggle/working').exists():
        downloaded = download_competition_data(Path('/kaggle/working/data'))
        if downloaded is not None:
            return downloaded

    available = []
    if input_root.exists():
        available = sorted(str(p) for p in input_root.glob('*'))[:30]
    raise FileNotFoundError(
        'Could not find competition CSV files. Attach playground-series-s6e7, '
        f'enable Kaggle API fallback, or provide local data/. Available /kaggle/input entries: {available}'
    )


DATA_DIR = find_data_dir()

WORK_DIR = Path('/kaggle/working')
if not WORK_DIR.exists():
    WORK_DIR = Path('.')

print('Data directory:', DATA_DIR.resolve())
print('Work directory:', WORK_DIR.resolve())

## 2. Load Data

The competition target is **`health_condition`**. We keep `id` only for the final submission and never use it as a predictive feature.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET = 'health_condition'
ID_COL = 'id'

feature_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]
X_raw = train[feature_cols].copy()
y = train[TARGET].copy()
X_test_raw = test[feature_cols].copy()

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Target distribution:')
display(y.value_counts(normalize=True).rename('share').to_frame().style.format({'share': '{:.2%}'}))

## 3. Domain-Ordered Feature Engineering

The strongest EDA signal is not a single raw column; it is the combination of **sleep**, **activity**, **stress**, **BMI**, and **lifestyle choices**.

For tree models that do not natively understand ordered categories, we add compact ordinal versions where the order has health meaning. These are not target encodings; they are **domain-ordered encodings** computed from row values only.

In [ ]:
ORDERED_MAPS = {
    'diet_type': {'veg': 0.0, 'non-veg': 1.0, 'balanced': 2.0},
    'stress_level': {'high': 0.0, 'medium': 1.0, 'low': 2.0},
    'sleep_quality': {'poor': 0.0, 'average': 1.0, 'good': 2.0},
    'physical_activity_level': {'sedentary': 0.0, 'moderate': 1.0, 'active': 2.0},
    'smoking_alcohol': {'yes': 0.0, 'occasional': 1.0, 'no': 2.0},
}


def _safe_num(df: pd.DataFrame, col: str, default=np.nan) -> pd.Series:
    if col in df.columns:
        return pd.to_numeric(df[col], errors='coerce')
    return pd.Series(default, index=df.index, dtype='float64')


def build_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()
    out['missing_count'] = out.isna().sum(axis=1)
    out['numeric_missing_count'] = out[numeric_cols].isna().sum(axis=1) if numeric_cols else 0

    ordinal_cols = []
    for col, mapping in ORDERED_MAPS.items():
        if col in out.columns:
            new_col = f'{col}_ord'
            out[new_col] = out[col].map(mapping).astype('float64')
            ordinal_cols.append(new_col)

    sleep = _safe_num(out, 'sleep_duration')
    steps = _safe_num(out, 'step_count')
    exercise = _safe_num(out, 'exercise_duration')
    bmi = _safe_num(out, 'bmi')
    calories = _safe_num(out, 'calorie_expenditure')
    heart_rate = _safe_num(out, 'heart_rate')

    sleep_quality_ord = out.get('sleep_quality_ord', pd.Series(np.nan, index=out.index))
    activity_ord = out.get('physical_activity_level_ord', pd.Series(np.nan, index=out.index))
    stress_ord = out.get('stress_level_ord', pd.Series(np.nan, index=out.index))

    if ordinal_cols:
        out['lifestyle_balance_score'] = out[ordinal_cols].mean(axis=1)
        out['lifestyle_low_signal_count'] = (out[ordinal_cols] <= 0.5).sum(axis=1)
        out['lifestyle_high_signal_count'] = (out[ordinal_cols] >= 1.5).sum(axis=1)

    out['sleep_recovery_score'] = np.exp(-((sleep - 8.0) ** 2) / 6.0) * (1.0 + 0.20 * sleep_quality_ord)
    out['activity_volume_score'] = np.log1p(steps.clip(lower=0)) * (1.0 + 0.20 * activity_ord) + exercise / 45.0
    out['stress_sleep_pressure'] = (2.0 - stress_ord) * (8.0 - sleep).clip(lower=0)
    out['bmi_activity_ratio'] = bmi / (np.log1p(steps.clip(lower=0)) + 1.0)
    out['calorie_per_bmi'] = calories / (bmi + EPS)
    out['steps_per_bmi'] = steps / (bmi + EPS)
    out['activity_intensity'] = steps / (exercise + 1.0)
    out['calorie_per_step'] = calories / (steps + 1.0)
    out['exercise_per_sleep'] = exercise / (sleep + EPS)
    out['steps_per_sleep'] = steps / (sleep + EPS)
    out['bmi_sleep_interaction'] = bmi * sleep

    out['low_sleep_flag'] = (sleep < 6.0).astype('float64')
    out['long_sleep_flag'] = (sleep > 9.0).astype('float64')
    out['bmi_risk_flag'] = ((bmi < 18.5) | (bmi >= 25.0)).astype('float64')
    out['cardio_rate_flag'] = ((heart_rate < 60.0) | (heart_rate > 100.0)).astype('float64')
    out['high_stress_low_sleep_flag'] = ((stress_ord <= 0.5) & (sleep < 6.5)).astype('float64')
    out['active_good_sleep_flag'] = ((activity_ord >= 1.5) & (sleep_quality_ord >= 1.5)).astype('float64')

    out = out.replace([np.inf, -np.inf], np.nan)
    return out


X_fe = build_domain_features(X_raw)
X_test_fe = build_domain_features(X_test_raw)

categorical_cols = X_fe.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = [c for c in X_fe.columns if c not in categorical_cols]
model_numeric_cols = X_fe.select_dtypes(include=[np.number, 'bool']).columns.tolist()

print('Engineered train shape:', X_fe.shape)
print('Categorical columns:', categorical_cols)
print('Numeric/model columns:', len(model_numeric_cols))
display(X_fe[model_numeric_cols].head())

## 4. Validation Utilities

Because our leaderboard evidence favors class-balanced behavior, champion selection prioritizes **balanced accuracy**, then **macro F1**, then ordinary accuracy.

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
le = LabelEncoder()
y_enc = le.fit_transform(y)
classes = list(le.classes_)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}


def prediction_mix(preds: np.ndarray) -> dict:
    counts = pd.Series(preds).value_counts(normalize=True)
    return {f'pred_{cls}_share': float(counts.get(cls, 0.0)) for cls in classes}


def metric_row(name: str, oof_pred: np.ndarray, test_pred: np.ndarray, notes: str = '') -> dict:
    row = {
        'candidate': name,
        'accuracy': accuracy_score(y, oof_pred),
        'balanced_accuracy': balanced_accuracy_score(y, oof_pred),
        'macro_f1': f1_score(y, oof_pred, average='macro'),
        'weighted_f1': f1_score(y, oof_pred, average='weighted'),
        'notes': notes,
    }
    row.update(prediction_mix(test_pred))
    return row


def normalize_rows(proba: np.ndarray) -> np.ndarray:
    proba = np.clip(proba, EPS, None)
    return proba / proba.sum(axis=1, keepdims=True)


def apply_class_multipliers(proba: np.ndarray, fit_multiplier: float = 1.0, unhealthy_multiplier: float = 1.0) -> np.ndarray:
    adjusted = proba.copy()
    if 'fit' in class_to_idx:
        adjusted[:, class_to_idx['fit']] *= fit_multiplier
    if 'unhealthy' in class_to_idx:
        adjusted[:, class_to_idx['unhealthy']] *= unhealthy_multiplier
    return normalize_rows(adjusted)


def proba_to_labels(proba: np.ndarray) -> np.ndarray:
    return le.inverse_transform(np.argmax(proba, axis=1))


def display_diagnostics(name: str, oof_pred: np.ndarray):
    print(f'===== {name} classification report =====')
    print(classification_report(y, oof_pred, digits=4))
    cm = pd.DataFrame(confusion_matrix(y, oof_pred, labels=classes), index=classes, columns=classes)
    print(f'===== {name} confusion matrix =====')
    display(cm)

## 5. Candidate 1: Balanced HGB Anchor

This is the continuity model from our earlier public runs. It gives us a **minority-sensitive anchor** so the new ensemble is not judged only against local accuracy.

In [ ]:
def make_hgb_pipeline() -> Pipeline:
    numeric_transformer = Pipeline(
        steps=[('imputer', SimpleImputer(strategy='median', add_indicator=True))]
    )
    categorical_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('cat', categorical_transformer, categorical_cols),
        ],
        remainder='drop',
    )
    model = HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_iter=250,
        l2_regularization=0.01,
        random_state=SEED,
    )
    return Pipeline(steps=[('preprocess', preprocessor), ('model', model)])


def evaluate_hgb_balanced():
    n_classes = len(classes)
    oof_proba = np.zeros((len(X_fe), n_classes), dtype='float64')
    test_proba = np.zeros((len(X_test_fe), n_classes), dtype='float64')

    for fold, (trn_idx, val_idx) in enumerate(cv.split(X_fe, y), 1):
        X_trn, X_val = X_fe.iloc[trn_idx], X_fe.iloc[val_idx]
        y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]
        sample_weight = compute_sample_weight(class_weight='balanced', y=y_trn)

        pipe = make_hgb_pipeline()
        pipe.fit(X_trn, y_trn, model__sample_weight=sample_weight)
        val_proba = pipe.predict_proba(X_val)
        test_proba_fold = pipe.predict_proba(X_test_fe)
        val_pred = proba_to_labels(val_proba)

        oof_proba[val_idx] = val_proba
        test_proba += test_proba_fold / N_SPLITS
        print(f'Fold {fold}: balanced accuracy = {balanced_accuracy_score(y_val, val_pred):.5f}')

    oof_proba = normalize_rows(oof_proba)
    test_proba = normalize_rows(test_proba)
    return proba_to_labels(oof_proba), proba_to_labels(test_proba), oof_proba, test_proba


hgb_oof, hgb_test_pred, hgb_oof_proba, hgb_test_proba = evaluate_hgb_balanced()
display_diagnostics('hgb_balanced_domain', hgb_oof)

## 6. Candidate 2: Balanced LGBM/XGB Domain Ensemble

The improvement idea is simple: use two strong gradient-boosted tree libraries with different inductive biases, then blend their probabilities.

Key differences from the public reference notebook:

- our feature names and composites follow our EDA narrative;
- the blend weight is selected from **OOF validation**, not hard-coded;
- we keep full diagnostics and write comparison artifacts for leaderboard discipline.

In [ ]:
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except Exception as exc:
    lgb = None
    LGBM_AVAILABLE = False
    print('LightGBM unavailable; continuing with XGBoost-only candidate.')
    print(type(exc).__name__, exc)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception as exc:
    xgb = None
    XGB_AVAILABLE = False
    print('XGBoost unavailable.')
    print(type(exc).__name__, exc)

if not XGB_AVAILABLE:
    raise ImportError('This notebook requires xgboost for the boosted-tree candidate.')

X_gbdt = X_fe[model_numeric_cols].astype('float64')
X_test_gbdt = X_test_fe[model_numeric_cols].astype('float64')


def make_lgbm_model():
    return lgb.LGBMClassifier(
        objective='multiclass',
        n_estimators=260,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )


def make_xgb_model():
    return xgb.XGBClassifier(
        objective='multi:softprob',
        n_estimators=260,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=2,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1,
    )


def evaluate_lgbm_xgb_ensemble():
    n_classes = len(classes)
    oof_lgb = np.zeros((len(X_gbdt), n_classes), dtype='float64') if LGBM_AVAILABLE else None
    oof_xgb = np.zeros((len(X_gbdt), n_classes), dtype='float64')
    test_lgb = np.zeros((len(X_test_gbdt), n_classes), dtype='float64') if LGBM_AVAILABLE else None
    test_xgb = np.zeros((len(X_test_gbdt), n_classes), dtype='float64')

    for fold, (trn_idx, val_idx) in enumerate(cv.split(X_gbdt, y), 1):
        X_trn, X_val = X_gbdt.iloc[trn_idx], X_gbdt.iloc[val_idx]
        y_trn_enc, y_val_enc = y_enc[trn_idx], y_enc[val_idx]
        y_val = y.iloc[val_idx]

        fold_parts = []
        if LGBM_AVAILABLE:
            lgb_model = make_lgbm_model()
            lgb_model.fit(X_trn, y_trn_enc)
            lgb_val = lgb_model.predict_proba(X_val)
            lgb_test = lgb_model.predict_proba(X_test_gbdt)
            oof_lgb[val_idx] = lgb_val
            test_lgb += lgb_test / N_SPLITS
            lgb_pred = le.inverse_transform(np.argmax(lgb_val, axis=1))
            fold_parts.append(f'LGBM bal_acc={balanced_accuracy_score(y_val, lgb_pred):.5f}')

        xgb_model = make_xgb_model()
        xgb_weight = compute_sample_weight(class_weight='balanced', y=y_trn_enc)
        xgb_model.fit(X_trn, y_trn_enc, sample_weight=xgb_weight, verbose=False)
        xgb_val = xgb_model.predict_proba(X_val)
        xgb_test_fold = xgb_model.predict_proba(X_test_gbdt)

        oof_xgb[val_idx] = xgb_val
        test_xgb += xgb_test_fold / N_SPLITS
        xgb_pred = le.inverse_transform(np.argmax(xgb_val, axis=1))
        fold_parts.append(f'XGB bal_acc={balanced_accuracy_score(y_val, xgb_pred):.5f}')
        print(f'Fold {fold}: ' + ', '.join(fold_parts))

    blend_rows = []
    weight_grid = [0.0, 0.15, 0.30, 0.50, 0.70, 0.85, 1.0] if LGBM_AVAILABLE else [0.0]
    for lgb_weight in weight_grid:
        blend = lgb_weight * oof_lgb + (1.0 - lgb_weight) * oof_xgb if LGBM_AVAILABLE else oof_xgb
        pred = le.inverse_transform(np.argmax(blend, axis=1))
        blend_rows.append(
            {
                'lgb_weight': lgb_weight,
                'xgb_weight': 1.0 - lgb_weight,
                'accuracy': accuracy_score(y, pred),
                'balanced_accuracy': balanced_accuracy_score(y, pred),
                'macro_f1': f1_score(y, pred, average='macro'),
                'weighted_f1': f1_score(y, pred, average='weighted'),
            }
        )

    blend_results = pd.DataFrame(blend_rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)
    best_weight = float(blend_results.loc[0, 'lgb_weight'])

    if LGBM_AVAILABLE:
        oof_blend = best_weight * oof_lgb + (1.0 - best_weight) * oof_xgb
        test_blend = best_weight * test_lgb + (1.0 - best_weight) * test_xgb
    else:
        oof_blend = oof_xgb
        test_blend = test_xgb

    oof_pred = le.inverse_transform(np.argmax(oof_blend, axis=1))
    test_pred = le.inverse_transform(np.argmax(test_blend, axis=1))

    return oof_pred, test_pred, normalize_rows(oof_blend), normalize_rows(test_blend), blend_results, best_weight


ensemble_oof, ensemble_test_pred, ensemble_oof_proba, ensemble_test_proba, blend_results, best_lgb_weight = evaluate_lgbm_xgb_ensemble()

print('Best OOF blend weight:')
print({'lgb_weight': best_lgb_weight, 'xgb_weight': 1.0 - best_lgb_weight})
display(blend_results)
display_diagnostics('lgbm_xgb_domain_ensemble', ensemble_oof)

## 7. HGB + LGBM/XGB Probability Blend

The v8 champion blends **LightGBM and XGBoost** only. The balanced HGB anchor uses a different preprocessing path and may capture complementary errors. We sweep a convex mix of HGB probabilities with the selected LGBM/XGB blend and keep the weight that maximizes OOF balanced accuracy.

In [ ]:
def sweep_hgb_lgbm_xgb_blend():
    rows = []
    hgb_weight_grid = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.85, 1.0]

    for hgb_weight in hgb_weight_grid:
        ensemble_weight = 1.0 - hgb_weight
        oof_blend = hgb_weight * hgb_oof_proba + ensemble_weight * ensemble_oof_proba
        test_blend = hgb_weight * hgb_test_proba + ensemble_weight * ensemble_test_proba
        oof_blend = normalize_rows(oof_blend)
        test_blend = normalize_rows(test_blend)
        oof_pred = proba_to_labels(oof_blend)
        test_pred = proba_to_labels(test_blend)
        rows.append(
            {
                'hgb_weight': hgb_weight,
                'ensemble_weight': ensemble_weight,
                'accuracy': accuracy_score(y, oof_pred),
                'balanced_accuracy': balanced_accuracy_score(y, oof_pred),
                'macro_f1': f1_score(y, oof_pred, average='macro'),
                'weighted_f1': f1_score(y, oof_pred, average='weighted'),
            }
        )

    hgb_blend_results = pd.DataFrame(rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)

    best_hgb_weight = float(hgb_blend_results.loc[0, 'hgb_weight'])
    best_ensemble_weight = 1.0 - best_hgb_weight
    best_oof_proba = normalize_rows(best_hgb_weight * hgb_oof_proba + best_ensemble_weight * ensemble_oof_proba)
    best_test_proba = normalize_rows(best_hgb_weight * hgb_test_proba + best_ensemble_weight * ensemble_test_proba)
    best_oof_pred = proba_to_labels(best_oof_proba)
    best_test_pred = proba_to_labels(best_test_proba)
    return hgb_blend_results, best_oof_pred, best_test_pred, best_hgb_weight


hgb_blend_results, hgb_ensemble_oof, hgb_ensemble_test_pred, best_hgb_weight = sweep_hgb_lgbm_xgb_blend()

print('Best HGB + LGBM/XGB blend weight:')
print({'hgb_weight': best_hgb_weight, 'ensemble_weight': 1.0 - best_hgb_weight})
display(hgb_blend_results)
hgb_blend_results.to_csv(WORK_DIR / 'hgb_ensemble_blend_results.csv', index=False)
display_diagnostics('hgb_lgbm_xgb_domain_blend', hgb_ensemble_oof)

## 8. Calibration And CatBoost Signal Engine

The v8 champion was selected from the **LGBM/XGB probability blend**. We test two conservative extensions:

1. **Class-prior calibration**: small multipliers for `fit` and `unhealthy`.
2. **CatBoost signal engine**: native-categorical CatBoost acts as a third opinion and may flip a tightly capped number of rows only when OOF validation supports the rule.

This adapts the useful **watchdog/arbitration** idea from the reference notebook while avoiding external submission pools and leaderboard-score dependencies.

In [ ]:
def sweep_class_prior_calibration():
    rows = []
    fit_multipliers = [0.88, 0.92, 0.96, 1.00, 1.04, 1.08, 1.12]
    unhealthy_multipliers = [0.88, 0.92, 0.96, 1.00, 1.04, 1.08, 1.12]

    for fit_multiplier in fit_multipliers:
        for unhealthy_multiplier in unhealthy_multipliers:
            adjusted_oof = apply_class_multipliers(ensemble_oof_proba, fit_multiplier, unhealthy_multiplier)
            adjusted_test = apply_class_multipliers(ensemble_test_proba, fit_multiplier, unhealthy_multiplier)
            oof_pred = proba_to_labels(adjusted_oof)
            test_pred = proba_to_labels(adjusted_test)
            row = metric_row(
                'calibrated_lgbm_xgb_domain_ensemble',
                oof_pred,
                test_pred,
                notes=(
                    f'LGBM/XGB probability blend; fit multiplier={fit_multiplier:.2f}; '
                    f'unhealthy multiplier={unhealthy_multiplier:.2f}.'
                ),
            )
            row.update(
                {
                    'fit_multiplier': fit_multiplier,
                    'unhealthy_multiplier': unhealthy_multiplier,
                }
            )
            rows.append(row)

    calibration_results = pd.DataFrame(rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)

    best = calibration_results.loc[0]
    best_oof_proba = apply_class_multipliers(ensemble_oof_proba, best['fit_multiplier'], best['unhealthy_multiplier'])
    best_test_proba = apply_class_multipliers(ensemble_test_proba, best['fit_multiplier'], best['unhealthy_multiplier'])

    best_oof_pred = proba_to_labels(best_oof_proba)
    best_test_pred = proba_to_labels(best_test_proba)
    return calibration_results, best_oof_pred, best_test_pred, best_oof_proba, best_test_proba


def prepare_catboost_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in categorical_cols:
        out[col] = out[col].fillna('__missing__').astype(str)
    return out


try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except Exception as exc:
    CatBoostClassifier = None
    CATBOOST_AVAILABLE = False
    print('CatBoost unavailable; skipping CatBoost signal engine.')
    print(type(exc).__name__, exc)


def make_catboost_model():
    return CatBoostClassifier(
        loss_function='MultiClass',
        eval_metric='Accuracy',
        iterations=160,
        depth=7,
        learning_rate=0.08,
        l2_leaf_reg=7.0,
        random_seed=SEED,
        random_strength=0.35,
        bootstrap_type='Bernoulli',
        subsample=0.85,
        auto_class_weights='Balanced',
        allow_writing_files=False,
        task_type='CPU',
        thread_count=-1,
        verbose=False,
    )


def evaluate_catboost_native():
    if not CATBOOST_AVAILABLE:
        return None, None, None, None

    X_cat = prepare_catboost_frame(X_fe)
    X_test_cat = prepare_catboost_frame(X_test_fe)
    n_classes = len(classes)
    oof_proba = np.zeros((len(X_cat), n_classes), dtype='float64')
    test_proba = np.zeros((len(X_test_cat), n_classes), dtype='float64')

    for fold, (trn_idx, val_idx) in enumerate(cv.split(X_cat, y), 1):
        X_trn, X_val = X_cat.iloc[trn_idx], X_cat.iloc[val_idx]
        y_trn_enc = y_enc[trn_idx]
        y_val = y.iloc[val_idx]

        model = make_catboost_model()
        model.fit(X_trn, y_trn_enc, cat_features=categorical_cols)
        val_proba = model.predict_proba(X_val)
        test_proba_fold = model.predict_proba(X_test_cat)

        oof_proba[val_idx] = val_proba
        test_proba += test_proba_fold / N_SPLITS

        val_pred = proba_to_labels(val_proba)
        print(f'Fold {fold}: CatBoost balanced accuracy = {balanced_accuracy_score(y_val, val_pred):.5f}')

    oof_proba = normalize_rows(oof_proba)
    test_proba = normalize_rows(test_proba)
    return proba_to_labels(oof_proba), proba_to_labels(test_proba), oof_proba, test_proba


def build_signal_predictions(base_proba, arbiter_proba, gate):
    max_flips = int(gate['max_flips'])
    base_idx = np.argmax(base_proba, axis=1)
    arbiter_idx = np.argmax(arbiter_proba, axis=1)
    row = np.arange(len(base_idx))

    base_conf = base_proba[row, base_idx]
    arbiter_conf = arbiter_proba[row, arbiter_idx]
    arbiter_margin = arbiter_conf - arbiter_proba[row, base_idx]

    candidate = (
        (arbiter_idx != base_idx) &
        (base_conf <= gate['max_base_conf']) &
        (arbiter_conf >= gate['min_arbiter_conf']) &
        (arbiter_margin >= gate['min_margin'])
    )

    score = 0.55 * arbiter_conf + 0.30 * np.clip(arbiter_margin, 0, 1) + 0.15 * (1.0 - base_conf)
    candidate_rows = np.flatnonzero(candidate)
    if max_flips > 0 and len(candidate_rows) > max_flips:
        candidate_rows = candidate_rows[np.argsort(-score[candidate_rows], kind='stable')[:max_flips]]

    final_idx = base_idx.copy()
    final_idx[candidate_rows] = arbiter_idx[candidate_rows]
    return le.inverse_transform(final_idx), len(candidate_rows)


def sweep_catboost_signal_engine():
    if not CATBOOST_AVAILABLE:
        empty = pd.DataFrame()
        return empty, ensemble_oof, ensemble_test_pred

    rows = []
    gates = []
    for max_base_conf in [0.65, 0.70]:
        for min_arbiter_conf in [0.62, 0.74]:
            for min_margin in [0.08, 0.16]:
                for max_flips in [0, 25, 100, 500]:
                    gates.append(
                        {
                            'max_base_conf': max_base_conf,
                            'min_arbiter_conf': min_arbiter_conf,
                            'min_margin': min_margin,
                            'max_flips': max_flips,
                        }
                    )

    for gate in gates:
        oof_pred, oof_flips = build_signal_predictions(ensemble_oof_proba, catboost_oof_proba, gate)
        test_pred, test_flips = build_signal_predictions(ensemble_test_proba, catboost_test_proba, gate)
        row = metric_row(
            'catboost_signal_engine',
            oof_pred,
            test_pred,
            notes=(
                f"OOF-gated micro-flips: max_base_conf={gate['max_base_conf']:.2f}; "
                f"min_cat_conf={gate['min_arbiter_conf']:.2f}; min_margin={gate['min_margin']:.2f}; "
                f"max_flips={gate['max_flips']}."
            ),
        )
        row.update(gate)
        row['oof_flips'] = oof_flips
        row['test_flips'] = test_flips
        rows.append(row)

    signal_results = pd.DataFrame(rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)

    best_gate = signal_results.loc[0, ['max_base_conf', 'min_arbiter_conf', 'min_margin', 'max_flips']].to_dict()
    best_oof_pred, _ = build_signal_predictions(ensemble_oof_proba, catboost_oof_proba, best_gate)
    best_test_pred, _ = build_signal_predictions(ensemble_test_proba, catboost_test_proba, best_gate)
    return signal_results, best_oof_pred, best_test_pred


calibration_results, calibrated_oof, calibrated_test_pred, calibrated_oof_proba, calibrated_test_proba = sweep_class_prior_calibration()

print('Top calibrated class-prior candidates:')
display(calibration_results.head(15))
calibration_results.to_csv(WORK_DIR / 'calibration_blend_results.csv', index=False)
display_diagnostics('calibrated_lgbm_xgb_domain_ensemble', calibrated_oof)

catboost_oof, catboost_test_pred, catboost_oof_proba, catboost_test_proba = evaluate_catboost_native()
if CATBOOST_AVAILABLE:
    display_diagnostics('catboost_native_balanced', catboost_oof)

signal_results, signal_oof, signal_test_pred = sweep_catboost_signal_engine()
if CATBOOST_AVAILABLE:
    print('Top CatBoost signal-engine candidates:')
    display(signal_results.head(15))
    signal_results.to_csv(WORK_DIR / 'catboost_signal_results.csv', index=False)
    display_diagnostics('catboost_signal_engine', signal_oof)
else:
    pd.DataFrame().to_csv(WORK_DIR / 'catboost_signal_results.csv', index=False)

results = [
    metric_row(
        'hgb_balanced_domain',
        hgb_oof,
        hgb_test_pred,
        notes='Continuity anchor with balanced sample weights and domain features.',
    ),
    metric_row(
        'lgbm_xgb_domain_ensemble',
        ensemble_oof,
        ensemble_test_pred,
        notes=f'Balanced LGBM/XGB probability blend selected by OOF sweep; LGBM weight={best_lgb_weight:.2f}.',
    ),
    metric_row(
        'hgb_lgbm_xgb_domain_blend',
        hgb_ensemble_oof,
        hgb_ensemble_test_pred,
        notes=(
            f'HGB + LGBM/XGB probability blend selected by OOF sweep; '
            f'HGB weight={best_hgb_weight:.2f}; ensemble weight={1.0 - best_hgb_weight:.2f}.'
        ),
    ),
    metric_row(
        'calibrated_lgbm_xgb_domain_ensemble',
        calibrated_oof,
        calibrated_test_pred,
        notes=str(calibration_results.loc[0, 'notes']),
    ),
]

candidate_predictions = {
    'hgb_balanced_domain': (hgb_oof, hgb_test_pred),
    'lgbm_xgb_domain_ensemble': (ensemble_oof, ensemble_test_pred),
    'hgb_lgbm_xgb_domain_blend': (hgb_ensemble_oof, hgb_ensemble_test_pred),
    'calibrated_lgbm_xgb_domain_ensemble': (calibrated_oof, calibrated_test_pred),
}

if CATBOOST_AVAILABLE:
    results.extend(
        [
            metric_row(
                'catboost_native_balanced',
                catboost_oof,
                catboost_test_pred,
                notes='Native-categorical CatBoost with balanced class weights.',
            ),
            metric_row(
                'catboost_signal_engine',
                signal_oof,
                signal_test_pred,
                notes=str(signal_results.loc[0, 'notes']),
            ),
        ]
    )
    candidate_predictions['catboost_native_balanced'] = (catboost_oof, catboost_test_pred)
    candidate_predictions['catboost_signal_engine'] = (signal_oof, signal_test_pred)

comparison = pd.DataFrame(results).sort_values(
    ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
).reset_index(drop=True)

BASE_CANDIDATE = 'lgbm_xgb_domain_ensemble'
MIN_BALANCED_ACCURACY_GAIN = 0.0002
base_row = comparison.loc[comparison['candidate'] == BASE_CANDIDATE].iloc[0]
comparison['balanced_accuracy_gain_vs_base'] = comparison['balanced_accuracy'] - float(base_row['balanced_accuracy'])
comparison['macro_f1_gain_vs_base'] = comparison['macro_f1'] - float(base_row['macro_f1'])
comparison['passes_champion_gate'] = (
    (comparison['candidate'] == BASE_CANDIDATE) |
    (
        (comparison['balanced_accuracy_gain_vs_base'] >= MIN_BALANCED_ACCURACY_GAIN) &
        (comparison['macro_f1_gain_vs_base'] >= 0.0)
    )
)

eligible = comparison[comparison['passes_champion_gate']].copy()
champion_name = eligible.iloc[0]['candidate']
champion_oof, champion_test_pred = candidate_predictions[champion_name]

print('Champion:', champion_name)
print('Champion gate: require >= 0.0002 balanced-accuracy gain and no macro-F1 loss versus v8 LGBM/XGB, unless using the v8 base itself.')
display(comparison)

comparison.to_csv(WORK_DIR / 'baseline_model_comparison.csv', index=False)
blend_results.to_csv(WORK_DIR / 'blend_weight_results.csv', index=False)
pd.DataFrame({'id': train[ID_COL], 'target': y, 'oof_pred': champion_oof}).to_csv(WORK_DIR / 'champion_oof_predictions.csv', index=False)

## 9. Champion Diagnostics

Before leaderboard submission, inspect whether the champion keeps a reasonable **prediction mix** and does not collapse minority classes.

In [ ]:
print(classification_report(y, champion_oof, digits=4))

cm = pd.DataFrame(confusion_matrix(y, champion_oof, labels=classes), index=classes, columns=classes)
display(cm)

mix = pd.DataFrame(
    {
        'train_target_share': y.value_counts(normalize=True),
        'champion_oof_share': pd.Series(champion_oof).value_counts(normalize=True),
        'champion_test_share': pd.Series(champion_test_pred).value_counts(normalize=True),
    }
).reindex(classes).fillna(0.0)

display(mix.style.format('{:.2%}'))

## 10. Create Notebook-Generated Submission

The submitted file should come from this public notebook output. This keeps the leaderboard record reproducible and easy to audit.

In [ ]:
submission = sample_submission.copy()
submission[TARGET] = champion_test_pred
submission_path = WORK_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)

print('Wrote:', submission_path)
display(submission.head())
display(submission[TARGET].value_counts(normalize=True).rename('submission_share').to_frame().style.format({'submission_share': '{:.2%}'}))

## 11. Submission Gate

Submit the notebook output only if these checks pass:

- `submission.csv` exists in `/kaggle/working`;
- selected champion is supported by **balanced accuracy** and **macro F1**, not only raw accuracy;
- `fit` and `unhealthy` predictions remain visible in the test mix;
- `baseline_model_comparison.csv`, `blend_weight_results.csv`, `hgb_ensemble_blend_results.csv`, `calibration_blend_results.csv`, and `catboost_signal_results.csv` explain why the champion was selected.

In [ ]:
assert submission_path.exists(), 'submission.csv was not created.'
assert len(submission) == len(test), 'Submission row count mismatch.'
assert submission[ID_COL].equals(sample_submission[ID_COL]), 'Submission IDs do not match sample submission.'
assert submission[TARGET].isna().sum() == 0, 'Submission contains missing predictions.'
assert set(submission[TARGET]).issubset(set(classes)), 'Submission contains unknown labels.'

print('Submission gate passed.')
print('Champion:', champion_name)
print('Public leaderboard anchor to beat: 0.94959')